# Ensemble Document Validation — Evaluation

Measures validation accuracy against labeled test data in `test_data/`.

Sections:
1. Setup
2. Ground truth loader
3. Batch eval run (with caching)
4. Metrics & confusion matrix
5. Detector ablation
6. Threshold calibration

## 1. Setup

In [ ]:
import dataclasses
import importlib.util
import subprocess
import sys
from pathlib import Path

import cv2
import fitz
import numpy as np

PROJECT_ROOT = Path.cwd()
SRC_PATH = str(PROJECT_ROOT / "src")

missing = [name for name in ("cv2", "fitz") if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".[dev]"])

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

for mod in list(sys.modules):
    if mod == "document_validation" or mod.startswith("document_validation."):
        del sys.modules[mod]

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

from document_validation import (
    ValidationConfig,
    probe_available_detectors,
    recompute_page_from_votes,
    validate_document_file_ensemble,
)
from document_validation.ensemble import OpenCvGateDetector
from document_validation.validator import _load_image, _render_pdf_pages

# Calibrated from eval data: OR blur logic, thresholds derived from feature percentile analysis.
# blur_laplacian_threshold / blur_tenengrad_threshold: equalized-image values that separate
#   blur files (p50 lap≈2300, p50 ten≈79) from accepted (p50 lap≈4000, p50 ten≈100).
# min_document_confidence: lowered from 0.95 → 0.75; accepted p25=0.90, not_document p50=0.70.
# min_readability_contrast: 65 sits between blur p50=57 and accepted p50=77.
# min_reject_confidence: 0.50 allows weak but consistent signals to register.
config = ValidationConfig(
    blur_laplacian_threshold=2000.0,
    blur_tenengrad_threshold=65.0,
    min_readability_contrast=65.0,
    max_low_readability_gray_std=65.0,
    min_document_confidence=0.75,
    min_reject_confidence=0.50,
    pdf_dpi=200,
)

requested_detectors = ["opencv", "doctr"]
detectors, detector_warnings = probe_available_detectors(requested_detectors)
for w in detector_warnings:
    print("Note:", w)

detector_importance = {
    "opencv": 1.0,
    "doctr": 1.25,
}

EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".pdf"}
LABELS = ["accepted", "blur", "cut", "not_document"]
TIE_BREAK = ("not_document", "blur", "not_clear", "cut", "accepted")

print("Using detectors:", detectors)
print("Ready")

## 2. Ground Truth Loader

Walks `test_data/<DocType>/<State>/<category>/` and maps folder names to expected labels.
Only `is_blur`, `is_not_full_document(cut)`, `is_not_document`, and `is_clear` are used.

In [ ]:
FOLDER_TO_LABEL = {
    "is_blur": "blur",
    "is_not_full_document(cut)": "cut",
    "is_not_document": "not_document",
    "is_clear": "accepted",
}

def load_ground_truth(test_data_dir: Path) -> pd.DataFrame:
    rows = []
    for doc_type_dir in sorted(test_data_dir.iterdir()):
        if not doc_type_dir.is_dir():
            continue
        for state_dir in sorted(doc_type_dir.iterdir()):
            if not state_dir.is_dir():
                continue
            for category_dir in sorted(state_dir.iterdir()):
                if not category_dir.is_dir():
                    continue
                label = FOLDER_TO_LABEL.get(category_dir.name)
                if label is None:
                    continue
                # rglob handles both flat folders and subdirs like is_blur/max/, is_blur/min/
                for f in sorted(category_dir.rglob("*")):
                    if f.is_file() and f.suffix.lower() in EXTENSIONS:
                        rows.append({
                            "file_path": f,
                            "doc_type": doc_type_dir.name,
                            "state": state_dir.name,
                            "expected_label": label,
                        })
    return pd.DataFrame(rows)

TEST_DATA_DIR = PROJECT_ROOT / "test_data"
gt_df = load_ground_truth(TEST_DATA_DIR)
print(f"Ground truth: {len(gt_df)} files")
gt_df.groupby(["doc_type", "state", "expected_label"]).size().unstack(fill_value=0)

## 3. Batch Eval Run

Runs the ensemble on every labeled file. Caches `EnsembleFileResult` objects and page
BGR images for reuse in threshold calibration (Section 6).

In [ ]:
def _file_label_from_issues(issues: list) -> str:
    # not_clear is treated as blur for eval: annotators use "blur" for both optical blur
    # and low-readability/washed-out documents, which the detector separates as not_clear.
    normalised = ["blur" if iss == "not_clear" else iss for iss in issues]
    if not normalised:
        return "accepted"
    for label in TIE_BREAK:
        if label in normalised:
            return label
    return normalised[0]


# RESULT_CACHE: str(file_path) -> EnsembleFileResult (contains all DetectorVote objects)
# PAGE_CACHE:   str(file_path) -> list of BGR page images (for OpenCV threshold sweeps)
RESULT_CACHE: dict = {}
PAGE_CACHE: dict = {}

eval_rows = []
n = len(gt_df)

for i, (_, row) in enumerate(gt_df.iterrows()):
    if (i + 1) % 50 == 0 or i == n - 1:
        print(f"  {i + 1}/{n}", end="\r")
    file_path = row["file_path"]
    key = str(file_path)
    try:
        result = validate_document_file_ensemble(
            file_path,
            config=config,
            detectors=detectors,
            detector_importance=detector_importance,
        )
        RESULT_CACHE[key] = result

        # Cache page images for OpenCV threshold sweeps
        if file_path.suffix.lower() == ".pdf":
            PAGE_CACHE[key] = list(_render_pdf_pages(file_path, config))
        else:
            PAGE_CACHE[key] = [_load_image(file_path)]

        predicted = _file_label_from_issues(result.issues)
        eval_rows.append({**row.to_dict(), "predicted_label": predicted, "page_count": result.page_count})
    except Exception as exc:
        eval_rows.append({**row.to_dict(), "predicted_label": "error", "page_count": 0, "error": str(exc)})

results_df = pd.DataFrame(eval_rows)
errors = (results_df["predicted_label"] == "error").sum()
print(f"\nDone. {len(results_df)} files evaluated. Errors: {errors}")
if errors:
    print(results_df[results_df["predicted_label"] == "error"][["file_path", "error"]])

## 4. Metrics & Confusion Matrix

Per-class precision/recall/F1, overall confusion matrix, and per-state breakdown.

In [ ]:
valid_df = results_df[results_df["predicted_label"] != "error"].copy()
y_true = valid_df["expected_label"]
y_pred = valid_df["predicted_label"]

print(f"Files evaluated: {len(valid_df)}")
print()
print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=LABELS)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(LABELS)))
ax.set_xticklabels(LABELS, rotation=45, ha="right")
ax.set_yticks(range(len(LABELS)))
ax.set_yticklabels(LABELS)
ax.set_xlabel("Predicted")
ax.set_ylabel("Expected")
ax.set_title("Confusion Matrix — Ensemble")
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
state_rows = []
for state, grp in valid_df.groupby("state"):
    yt = grp["expected_label"]
    yp = grp["predicted_label"]
    row = {
        "state": state,
        "n_files": len(grp),
        "accuracy": round((yt == yp).mean(), 3),
    }
    for cls in LABELS:
        if (yt == cls).any():
            row[f"{cls}_F1"] = round(f1_score(yt == cls, yp == cls, zero_division=0), 3)
        else:
            row[f"{cls}_F1"] = float("nan")
    state_rows.append(row)

state_df = pd.DataFrame(state_rows).set_index("state").sort_values("accuracy")
f1_cols = [c for c in state_df.columns if "F1" in c]
state_df.style.format("{:.3f}", subset=f1_cols + ["accuracy"]).background_gradient(
    cmap="RdYlGn", subset=f1_cols + ["accuracy"]
)

## 5. Detector Ablation

Runs each detector in isolation and compares precision/recall/F1 with the full ensemble.
Shows whether doctr adds value over opencv alone and for which issue classes.
Note: this re-runs validation, so it takes ~same time as Section 3.

In [ ]:
ablation_configs = {
    "opencv": ["opencv"],
    "doctr": [d for d in ["doctr"] if d in detectors],  # skip if doctr not installed
    "ensemble": detectors,
}

ablation_results: dict = {}

for det_name, det_list in ablation_configs.items():
    if not det_list:
        print(f"Skipping {det_name}: not available")
        continue
    rows = []
    for i, (_, row) in enumerate(gt_df.iterrows()):
        if (i + 1) % 50 == 0:
            print(f"  {det_name}: {i + 1}/{len(gt_df)}", end="\r")
        try:
            result = validate_document_file_ensemble(
                row["file_path"],
                config=config,
                detectors=det_list,
                detector_importance=detector_importance,
            )
            predicted = _file_label_from_issues(result.issues)
            rows.append({**row.to_dict(), "predicted_label": predicted})
        except Exception:
            rows.append({**row.to_dict(), "predicted_label": "error"})
    ablation_results[det_name] = pd.DataFrame(rows)
    print(f"\n  {det_name} done.")

In [ ]:
ablation_rows = []
for det_name, df in ablation_results.items():
    valid = df[df["predicted_label"] != "error"]
    yt = valid["expected_label"]
    yp = valid["predicted_label"]
    for cls in LABELS:
        ablation_rows.append({
            "detector": det_name,
            "class": cls,
            "precision": round(precision_score(yt == cls, yp == cls, zero_division=0), 3),
            "recall":    round(recall_score(yt == cls, yp == cls, zero_division=0), 3),
            "f1":        round(f1_score(yt == cls, yp == cls, zero_division=0), 3),
        })

ablation_df = pd.DataFrame(ablation_rows)
ablation_pivot = ablation_df.pivot_table(
    index="class", columns="detector", values=["precision", "recall", "f1"]
).round(3)
ablation_pivot

## 6. Threshold Calibration

Sweeps three key ValidationConfig thresholds and plots per-class precision/recall/F1.
A vertical marker shows where the current config sits on each curve.

- `min_reject_confidence`: replayed from cached votes — no detector re-run
- `blur_laplacian_threshold`: re-runs OpenCV only; reuses cached doctr votes
- `min_document_confidence`: re-runs OpenCV only; reuses cached doctr votes

In [ ]:
import numpy as np


def _sweep_metrics(gt_df, pred_fn) -> pd.DataFrame:
    rows = []
    for _, row in gt_df.iterrows():
        try:
            rows.append({"expected_label": row["expected_label"], "predicted_label": pred_fn(row)})
        except Exception:
            rows.append({"expected_label": row["expected_label"], "predicted_label": "error"})
    df = pd.DataFrame(rows)
    return df[df["predicted_label"] != "error"]


def _compute_class_metrics(valid_df, threshold_value) -> list:
    yt = valid_df["expected_label"]
    yp = valid_df["predicted_label"]
    return [
        {
            "threshold": threshold_value,
            "class": cls,
            "precision": round(precision_score(yt == cls, yp == cls, zero_division=0), 4),
            "recall":    round(recall_score(yt == cls, yp == cls, zero_division=0), 4),
            "f1":        round(f1_score(yt == cls, yp == cls, zero_division=0), 4),
        }
        for cls in LABELS
    ]


def _plot_threshold_sweep(sweep_df, title, current_value, xlabel):
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
    axes = axes.flatten()
    for idx, cls in enumerate(LABELS):
        ax = axes[idx]
        sub = sweep_df[sweep_df["class"] == cls]
        ax.plot(sub["threshold"], sub["precision"], marker="o", label="precision")
        ax.plot(sub["threshold"], sub["recall"],    marker="s", label="recall")
        ax.plot(sub["threshold"], sub["f1"],        marker="^", linewidth=2, label="F1")
        ax.axvline(x=current_value, color="gray", linestyle="--", alpha=0.7, label="current")
        ax.set_title(f"class: {cls}")
        ax.set_xlabel(xlabel)
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
MRC_VALUES = np.round(np.arange(0.30, 0.96, 0.05), 2)
mrc_rows = []

for thresh in MRC_VALUES:
    new_config = dataclasses.replace(config, min_reject_confidence=thresh)

    def _pred(row, nc=new_config):
        cached = RESULT_CACHE.get(str(row["file_path"]))
        if cached is None:
            raise ValueError("not in cache")
        new_pages = [
            recompute_page_from_votes(page.page_number, page.votes, nc)
            for page in cached.pages
        ]
        return _file_label_from_issues([iss for p in new_pages for iss in p.issues])

    valid = _sweep_metrics(gt_df, _pred)
    mrc_rows.extend(_compute_class_metrics(valid, thresh))

mrc_df = pd.DataFrame(mrc_rows)
_plot_threshold_sweep(
    mrc_df,
    title="Threshold sweep: min_reject_confidence",
    current_value=config.min_reject_confidence,
    xlabel="min_reject_confidence",
)

macro_mrc = mrc_df.groupby("threshold")["f1"].mean().reset_index()
best_mrc = macro_mrc.loc[macro_mrc["f1"].idxmax()]
print(f"Best macro-F1 for min_reject_confidence: {best_mrc['f1']:.4f} at threshold={best_mrc['threshold']:.2f}")
print(f"Current config: min_reject_confidence={config.min_reject_confidence}")

In [ ]:
BLT_VALUES = np.arange(20, 151, 10, dtype=float)
opencv_detector = OpenCvGateDetector()
blt_rows = []

for thresh in BLT_VALUES:
    new_config = dataclasses.replace(config, blur_laplacian_threshold=thresh)

    def _pred(row, nc=new_config):
        key = str(row["file_path"])
        cached = RESULT_CACHE.get(key)
        page_images = PAGE_CACHE.get(key)
        if cached is None or page_images is None:
            raise ValueError("not in cache")
        new_pages = []
        for page, page_bgr in zip(cached.pages, page_images):
            new_opencv = opencv_detector.detect(page_bgr, row["file_path"], page.page_number, nc)
            orig_opencv = next((v for v in page.votes if v.detector == "opencv_gate"), None)
            importance = orig_opencv.importance if orig_opencv is not None else 1.0
            new_opencv = dataclasses.replace(new_opencv, importance=importance)
            other_votes = [v for v in page.votes if v.detector != "opencv_gate"]
            new_votes = [new_opencv] + other_votes
            new_pages.append(recompute_page_from_votes(page.page_number, new_votes, nc))
        return _file_label_from_issues([iss for p in new_pages for iss in p.issues])

    valid = _sweep_metrics(gt_df, _pred)
    blt_rows.extend(_compute_class_metrics(valid, thresh))

blt_df = pd.DataFrame(blt_rows)
_plot_threshold_sweep(
    blt_df,
    title="Threshold sweep: blur_laplacian_threshold",
    current_value=config.blur_laplacian_threshold,
    xlabel="blur_laplacian_threshold",
)

macro_blt = blt_df.groupby("threshold")["f1"].mean().reset_index()
best_blt = macro_blt.loc[macro_blt["f1"].idxmax()]
print(f"Best macro-F1 for blur_laplacian_threshold: {best_blt['f1']:.4f} at threshold={best_blt['threshold']:.0f}")
print(f"Current config: blur_laplacian_threshold={config.blur_laplacian_threshold}")

In [ ]:
MDC_VALUES = np.round(np.arange(0.40, 0.96, 0.05), 2)
mdc_rows = []

for thresh in MDC_VALUES:
    new_config = dataclasses.replace(config, min_document_confidence=thresh)

    def _pred(row, nc=new_config):
        key = str(row["file_path"])
        cached = RESULT_CACHE.get(key)
        page_images = PAGE_CACHE.get(key)
        if cached is None or page_images is None:
            raise ValueError("not in cache")
        new_pages = []
        for page, page_bgr in zip(cached.pages, page_images):
            new_opencv = opencv_detector.detect(page_bgr, row["file_path"], page.page_number, nc)
            orig_opencv = next((v for v in page.votes if v.detector == "opencv_gate"), None)
            importance = orig_opencv.importance if orig_opencv is not None else 1.0
            new_opencv = dataclasses.replace(new_opencv, importance=importance)
            other_votes = [v for v in page.votes if v.detector != "opencv_gate"]
            new_votes = [new_opencv] + other_votes
            new_pages.append(recompute_page_from_votes(page.page_number, new_votes, nc))
        return _file_label_from_issues([iss for p in new_pages for iss in p.issues])

    valid = _sweep_metrics(gt_df, _pred)
    mdc_rows.extend(_compute_class_metrics(valid, thresh))

mdc_df = pd.DataFrame(mdc_rows)
_plot_threshold_sweep(
    mdc_df,
    title="Threshold sweep: min_document_confidence",
    current_value=config.min_document_confidence,
    xlabel="min_document_confidence",
)

macro_mdc = mdc_df.groupby("threshold")["f1"].mean().reset_index()
best_mdc = macro_mdc.loc[macro_mdc["f1"].idxmax()]
print(f"Best macro-F1 for min_document_confidence: {best_mdc['f1']:.4f} at threshold={best_mdc['threshold']:.2f}")
print(f"Current config: min_document_confidence={config.min_document_confidence}")

### Recommended Config

The cells below summarise the best-performing threshold value for each swept parameter.
Combine these only if improvements are independent — validate the combined config with
a fresh run of Section 3.

In [ ]:
print("Threshold calibration summary")
print("=" * 45)
print(f"  min_reject_confidence:    {config.min_reject_confidence:.2f}  →  best: {best_mrc['threshold']:.2f}  (macro-F1: {best_mrc['f1']:.4f})")
print(f"  blur_laplacian_threshold: {config.blur_laplacian_threshold:.0f}   →  best: {best_blt['threshold']:.0f}   (macro-F1: {best_blt['f1']:.4f})")
print(f"  min_document_confidence:  {config.min_document_confidence:.2f}  →  best: {best_mdc['threshold']:.2f}  (macro-F1: {best_mdc['f1']:.4f})")
print()
print("Suggested config (validate with a fresh Section 3 run):")
print(f"""
config = ValidationConfig(
    blur_laplacian_threshold={best_blt['threshold']:.0f},
    blur_tenengrad_threshold={config.blur_tenengrad_threshold},
    min_readability_contrast={config.min_readability_contrast},
    max_low_readability_gray_std={config.max_low_readability_gray_std},
    min_document_confidence={best_mdc['threshold']:.2f},
    min_reject_confidence={best_mrc['threshold']:.2f},
    pdf_dpi={config.pdf_dpi},
)""")